# Quantum chemistry in NanoVer: dynamic bond-forming and -breaking

In this notebook, we demonstrate how the ASESimulation class can be used to perform quantum chemical simulations that allow bonds to be formed and broken, and how this class can be used to visualise such processes via its dynamic bonding flag.

<div style="border-left: 4px solid #bc8cff;
            background-color: #352447;
            padding: 12px 16px;
            border-radius: 4px;
            color: #f0e6ff;">

<strong style="color: #8957e5;">IMPORTANT</strong>

<p>
To run this notebook, you will need to install the
<code>tblite-python</code> package that enables use of
Grimme's GFN2-xTB method as the ASE calculator
for the simulation.
</p>

</div>

## CH$_{3}$F + $^{-}$OH: Simulating an S$_{\rm{N}}$2 reaction using GFN2-xTB

The system we have chosen to demonstrate dynamic bonding in NanoVer comprises a methyl fluoride molecule and a hydroxide ion, which can undergo an S$_{\rm{N}}$2 reaction to form methanol and fluoride:

$$ \mathrm{CH}_{3}\mathrm{F} + ^{-}\mathrm{OH} \rightarrow \mathrm{CH}_{3}\mathrm{OH} + \mathrm{F}^{-} $$

We choose to simulate this reaction using the extended tight-binding method GFN2-xTB, developed by the group of Steffan Grimme [[1]](#ref1), which can be used as a calculator in ASE via the `xtb-python` library.


In [1]:
from ase.io import read
from tblite.ase import TBLite

First, let's set up the system, using an .xyz file to load the geometry of the nuclei, specifying the charge of the system, and setting the ASE calculator to use GFN2-xTB.

In [2]:
# Read the system geometry from .xyz
system = "CH3F_OH"
atoms = read("../systems/ase/CH3F_OH.xyz")
atoms.set_initial_charges([0., 0., 0., 0., 0., -1., 0.])

# Define GFNn-xTB method
atoms.calc = TBLite(method="GFN2-xTB", verbosity=0)

Now let's set up the NanoVer server and ASESimulation. When we create the ASESimulation, we set `dynamic_bonds=True`: this is what allows us to visualise the changes in molecular topology as the simulation progresses

In [3]:
from ase import units
from ase.md import Langevin
from nanover.ase import ASESimulation
from nanover.app import OmniRunner

In [4]:
# Set up the dynamics
dyn = Langevin(atoms, timestep=1 * units.fs, temperature_K=300, friction=0.5)

# Set up ASE simulation
ase_sim = ASESimulation.from_ase_dynamics(dyn)
ase_sim.dynamic_bonds = True
ase_sim.frame_interval = 1

# Check the simulation runs correctly
ase_sim.dynamics.run(10)
print(ase_sim.dynamics.nsteps)
print(ase_sim.dynamics.atoms.get_potential_energy())

C:\Users\ragzo\miniforge3\envs\nanover-dev\Lib\site-packages\ase\md\langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


10
-355.89944949831005


In [5]:
# Close any existing server from previous runs
OmniRunner.close_all_runners()

# Create the server and load the simulation in a paused state
imd_runner = OmniRunner.with_basic_server(ase_sim, name="Dynamic bonding server")
imd_runner.load(0)
imd_runner.pause()

## Visualising dynamic bonding: hyperballs

Now we have a simulation running that we can connect to and perform reactions in. However, the default ball and stick renderers are not as suitable for dynamic bonds, as their binary on/off nature fails to convey the more gradual change in electronic density as bonds are formed and broken. For this reason, we will use the hyperballs renderer, which provides a more continuous and fluid method to visualise such changes in the molecular system.

<div class="alert alert-warning">
WARNING:
The hyperballs renderer does not provide a representation of the actual electron densities which define the bonding of quantum chemical systems. It is important not to confuse this visual representation with the true physical quantities governing the simulation. Nevertheless, hyperballs can provide a useful proxy for how the bonding in the molecule changes as the reaction proceeds.
</div>


In [6]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.selections.update_selection("root", renderer={"color": {'type': "cpk", 'scheme': "nanover"}, "render" : "hyperballs", "tension" : 0.8, "scale" : 0.11})

## Explore in XR

That's it! You can now connect to the server and see if you can induce an S$_{\mathrm{N}}$2 reaction between the species. Remember to press the play button to start the simulation!

See if you can induce the reaction above via an S$_{\mathrm{N}}$2 mechanism. Here are some things to think about while you try this:
- What seems difficult about this?
- Can you also induce the reverse reaction?
- How do the forward and reverse reactions compare?
- What might that be telling you about this system?

## Closing the server

As ever, once we have finished with the simulation, we close the server using the cell below.

In [7]:
imd_runner.close()

## References

<a id="ref1"></a>
[1] Bannwarth, C., Ehlert, S., & Grimme, S. (2019). GFN2-xTB—An accurate and broadly parametrized self-consistent tight-binding quantum chemical method with multipole electrostatics and density-dependent dispersion contributions. Journal of chemical theory and computation, 15(3), 1652-1671.